In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import sys, os
import numpy as np

# Add the project root directory to the path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.pnn import PolynomialNeuralNetwork
from src.utils import save_model
from src.hc import compute_robust_radius, load_robust_radius_results, verify_experiment
from src.data import generate_sinusoidal_data, visualize_decision_boundary
from src.utils import train_epochs

In [ ]:
X_train, y_train = generate_sinusoidal_data(n_samples=2000, noise_level=0.1)

In [ ]:
model = PolynomialNeuralNetwork(
    input_dim=2,
    output_dim=2,
    hidden_dims=[8, 6],
    act_degree=3,
    homogeneous=False,
    bias=True,
    s=0.1,
)

In [ ]:
# Create DataLoader for training
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

In [ ]:
# Train the model
history = train_epochs(
    model=model,
    train_loader=train_loader,
    num_epochs=1000,
    optimizer_type="adam",
    learning_rate=1e-3,
    verbose=True,
)

In [ ]:
model = model.cpu()
X_train = X_train.cpu()
y_train = y_train.cpu()
visualize_decision_boundary(
    model,
    X_train,
    y_train,
    grid_size=1000,
    x_range=(-10, 10),
    y_range=(-10, 10),
)

In [ ]:
path = save_model(
    model,
    metadata={"description": "sinusoidal data"},
)


In [ ]:
xi_list=np.array([[-3.0, 1.0], [4.0, -2.0], [5.0, 5.0]])
res = compute_robust_radius(
    experiment_path=path,
    xi_list=xi_list,
    verbose=False,
    save_results=True,
    save_detailed=True,
)

In [ ]:
res

In [ ]:
model(torch.Tensor(res['closest_sol']))

In [ ]:
import plotly.graph_objects as go

In [ ]:
x_grid = np.linspace(-12, 12, 400)
y_grid = np.linspace(-12, 12, 400)
X, Y = np.meshgrid(x_grid, y_grid)
Z = model(torch.tensor(np.c_[X.ravel(), Y.ravel()], dtype=torch.float32))
Z = Z.detach().numpy()
Z = Z[:, 1] - Z[:, 0]
Z = Z.reshape(X.shape)

In [ ]:
fig = go.Figure()
fig.add_trace(
go.Contour(
    x=x_grid,
    y=y_grid,
    z=Z,
    contours=dict(start=0, end=0, coloring="lines"),
    colorscale=[[0, "blue"], [1, "blue"]],
    showscale=False,
    line=dict(width=3),
    name="Decision Boundary",
    showlegend=True,
    )
)

fig.add_trace(
    go.Scatter(
        x= xi_list[:,0],
        y= xi_list[:,1],
        mode="markers",
        marker=dict(color="red", size=6),
        name="Test Center Point",
    )
)

fig.add_trace(
    go.Scatter(
        x=res['closest_sol'][:,0],
        y=res['closest_sol'][:,1],
        mode="markers",
        marker=dict(color="green", size=6),
        name="The Closest Critical Point",
    )
)

for i in range(xi_list.shape[0]):
    fig.add_trace(
        go.Scatter(
            x=[xi_list[i, 0], res['closest_sol'][i, 0]],
            y=[xi_list[i, 1], res['closest_sol'][i, 1]],
            mode="lines",
            line=dict(color="grey", dash="dashdot", width=1),
            showlegend=False,
        )
    )

    # add circle to indicate robust radius
    # the radius is res['min_dist'][i]
    fig.add_shape(
        type="circle",
        xref="x",
        yref="y",
        x0=xi_list[i, 0] - res['min_dist'][i],
        y0=xi_list[i, 1] - res['min_dist'][i],
        x1=xi_list[i, 0] + res['min_dist'][i],
        y1=xi_list[i, 1] + res['min_dist'][i],
        line=dict(color="orange", dash="dot"),
    )

fig.update_layout(
    margin=dict(l=10, r=10, t=10, b=10),
    legend=dict(
        x=0.95,
        y=0.15,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.6)",  # optional: semi-transparent background
        bordercolor="black",
        borderwidth=1,
    ),
    xaxis=dict(showgrid=False, zeroline=False),
    yaxis=dict(showgrid=False, zeroline=False),
    xaxis_title="x0",
    yaxis_title="x1",
    width=800,
    height=800,
)

fig.show()